# Milestone 1: Data Collection, Exploration, and Preprocessing

**Objective:** Collect and prepare a high-quality dataset for training the facial recognition model.

## Outline
1. Dataset loading (LFW / VGGFace)
2. Identity distribution analysis
3. Image quality assessment
4. Preprocessing pipeline (resize → normalize → face crop → augment)
5. Save preprocessed data

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.data_collection import list_identities, collect_image_paths
from src.data.preprocessing import preprocess_image, augment_image

## 1. Dataset Overview

In [ ]:
raw_dir = Path('../data/raw')
identities = list_identities(raw_dir)
image_paths = collect_image_paths(raw_dir)

print(f'Total identities : {len(identities)}')
print(f'Total images     : {len(image_paths)}')

## 2. Identity Distribution

In [ ]:
from collections import Counter

counts = Counter(p.parent.name for p in image_paths)
top_n = 20
top_identities = counts.most_common(top_n)

names, values = zip(*top_identities) if top_identities else ([], [])
plt.figure(figsize=(14, 5))
plt.bar(names, values)
plt.xticks(rotation=45, ha='right')
plt.title(f'Top {top_n} Identities by Image Count')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

## 3. Sample Images

In [ ]:
import cv2

sample_paths = image_paths[:9]
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, path in zip(axes.flat, sample_paths):
    img = cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(path.parent.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline

In [ ]:
if image_paths:
    raw = cv2.imread(str(image_paths[0]))
    preprocessed = preprocess_image(raw, detect_face=True)
    if preprocessed is not None:
        plt.figure(figsize=(6, 3))
        plt.subplot(1, 2, 1)
        plt.imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB))
        plt.title('Original')
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.imshow(preprocessed)
        plt.title('Preprocessed (224×224, normalised)')
        plt.axis('off')
        plt.show()
    else:
        print('No face detected in sample image.')

## 5. Data Augmentation

In [ ]:
if image_paths:
    raw = cv2.imread(str(image_paths[0]))
    augmented = augment_image(raw)
    titles = ['Original', 'Horizontal Flip', 'Rotation +15°', 'Rotation −15°']
    images = [raw] + augmented
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()